# 🎓 SNBP 2026 PTN Data Scraper
Notebook ini akan melakukan kompilasi data daya tampung SNBP 2026 dan peminat 2025 secara multithreading hybrid API dari portal SNPMB.

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
import pandas as pd
import json
import os
from datetime import datetime
from tqdm import tqdm
import re
import concurrent.futures

BASE_PTN_URL = "https://sidata-ptn.snpmb.id/ptn_sn.php"
BASE_PRODI_URL = "https://sidatagrun-public-1076756628210.asia-southeast2.run.app/ptn_sn.php"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://snpmb.id/"
}

def get_session():
    session = requests.Session()
    retry = Retry(
        total=5, 
        backoff_factor=1,
        status_forcelist=[403, 429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(HEADERS)
    return session

In [ ]:
def get_ptn_mapping(session):
    """Ekstrak Master PTN Code & Name"""
    print("🔍 Mengambil Master Data PTN...")
    mapping = {}
    urls = [
        f"{BASE_PTN_URL}?ptn=-1",
        f"{BASE_PTN_URL}?ptn=-2",
        f"{BASE_PTN_URL}?ptn=-3"
    ]
    try:
        for u in urls:
            response = session.get(u, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            for tr in soup.select("table tbody tr"):
                cols = tr.find_all("td")
                if len(cols) >= 3:
                    kode_asli = cols[1].get_text(strip=True)
                    nama_raw = cols[2].get_text(separator=' ', strip=True)
                    nama_ptn = nama_raw.split('(')[0].strip()
                    a = tr.find("a", href=re.compile(r"ptn="))
                    if a:
                        href_param = a.get("href").split("ptn=")[1]
                        mapping[href_param] = {"kode": kode_asli, "nama": nama_ptn}
        print(f"✅ Berhasil! Ditemukan {len(mapping)} PTN \n")
        return mapping
    except Exception as e:
        print(f"❌ Gagal mengambil daftar master PTN: {e}")
        return {}

def process_ptn(href_param, ptn_info, session, csv_dir, json_dir):
    """Scraping tabel detil prodi dari Cloud Run API"""
    url = f"{BASE_PRODI_URL}?ptn={href_param}"
    ptn_data = []
    kode_ptn_asli = ptn_info["kode"]
    nama_ptn = ptn_info["nama"]
    try:
        response = session.get(url, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        table_rows = soup.select("table tbody tr")
        
        safe_ptn_name = re.sub(r'[^\w\s-]', '', nama_ptn.replace(" ", "_").replace("/", "-"))[:60]
        file_prefix = f"{kode_ptn_asli}_{safe_ptn_name}"
        
        if not table_rows:
            return {"kode": kode_ptn_asli, "nama": nama_ptn, "data": [], "status": "empty", "error": None}
            
        for idx, tr in enumerate(table_rows, start=1):
            cols = tr.find_all("td")
            if len(cols) < 6: continue
            prodi_record = {
                "NO": idx,
                "KODE_PTN": kode_ptn_asli,
                "NAMA_PTN": nama_ptn,
                "KODE_PRODI": cols[1].get_text(strip=True),
                "NAMA_PRODI": cols[2].get_text(strip=True),
                "JENJANG": cols[3].get_text(strip=True),
                "DAYA_TAMPUNG_2026": cols[4].get_text(strip=True),
                "PEMINAT_2025": cols[5].get_text(strip=True),
                "JENIS_PORTOFOLIO": cols[6].get_text(strip=True) if len(cols) > 6 else ""
            }
            ptn_data.append(prodi_record)
        
        if ptn_data:
            df_ptn = pd.DataFrame(ptn_data)
            df_ptn.to_csv(os.path.join(csv_dir, f"{file_prefix}.csv"), index=False, encoding='utf-8')
            with open(os.path.join(json_dir, f"{file_prefix}.json"), 'w', encoding='utf-8') as f:
                json.dump(ptn_data, f, indent=2, ensure_ascii=False)
                
        return {"kode": kode_ptn_asli, "nama": nama_ptn, "data": ptn_data, "status": "success", "error": None}
    except requests.exceptions.RequestException as e:
        return {"kode": kode_ptn_asli, "nama": nama_ptn, "data": [], "status": "error", "error": f"Connection Error: {str(e)}"}
    except Exception as e:
        return {"kode": kode_ptn_asli, "nama": nama_ptn, "data": [], "status": "error", "error": f"General Error: {str(e)}"}

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"scrape_results_{timestamp}"
csv_dir = os.path.join(output_dir, "csv")
json_dir = os.path.join(output_dir, "json")

os.makedirs(csv_dir, exist_ok=True)
os.makedirs(json_dir, exist_ok=True)

print("================================================================================")
print("🚀 ADVANCED PRODI SCRAPER - ORGANIZED BY PTN (MAXIMAL & POWERFUL)")
print("================================================================================")

session = get_session()
ptn_mapping = get_ptn_mapping(session)

if ptn_mapping:
    print(f"📊 Scraping {len(ptn_mapping)} PTN \n")
    all_prodi_summary = []
    stats = {"success": 0, "empty": 0, "error": 0}
    error_logs = []
    
    with tqdm(total=len(ptn_mapping), desc="Scraping Progress") as pbar:
        with concurrent.futures.ThreadPoolExecutor(max_workers=min(15, len(ptn_mapping))) as executor:
            future_to_ptn = {
                executor.submit(process_ptn, href_param, info, session, csv_dir, json_dir): href_param 
                for href_param, info in ptn_mapping.items()
            }
            
            for future in concurrent.futures.as_completed(future_to_ptn):
                try:
                    result = future.result()
                    if result["status"] == "success":
                        stats["success"] += 1
                        all_prodi_summary.extend(result["data"])
                    elif result["status"] == "empty":
                        stats["empty"] += 1
                    else:
                        stats["error"] += 1
                        error_logs.append(f"PTN {result['kode']}: {result['error']}")
                except Exception as exc:
                    stats["error"] += 1
                finally:
                    pbar.update(1)

    print("\n✅ SCRAPING COMPLETE!")
    print(f"📈 Total prodi dicatat: {len(all_prodi_summary)}")
    
    if all_prodi_summary:
        df_all = pd.DataFrame(all_prodi_summary)
        df_all['KODE_PTN'] = pd.to_numeric(df_all['KODE_PTN'], errors='coerce')
        df_all = df_all.sort_values(by=['KODE_PTN', 'NO']).reset_index(drop=True)
        df_all['KODE_PTN'] = df_all['KODE_PTN'].astype(str)
        
        df_all.to_csv(os.path.join(output_dir, "MASTER_all_prodi.csv"), index=False, encoding='utf-8')
        df_all.to_json(os.path.join(output_dir, "MASTER_all_prodi.json"), orient='records', indent=2, force_ascii=False)
        print("📁 MASTER Data tersimpan di folder:", output_dir)
else:
    print("Gagal mengambil metadata PTN.")

## Preview Ekstrak

In [ ]:
if 'df_all' in locals() and not df_all.empty:
    display(df_all.head())